# SeaVigil: Sentinel-1 vessel detection (Allen Institute model)

Runs the open, pre-trained **Allen Institute** Sentinel-1 vessel detector (the model behind Skylight, Apache-2.0) on a Copernicus scene, on a free Colab GPU. Output is a `detections.csv` (lat/lon, length, heading, speed, fishing-vessel class) that feeds SeaVigil's explanation + authorization + evidence layer.

**Before you run:**
1. `Runtime` menu -> `Change runtime type` -> **GPU**.
2. Click the **key icon** (left sidebar) -> add two secrets with notebook access ON:
   - `CDSE_S3_KEY`  = your Copernicus S3 **access key**
   - `CDSE_S3_SECRET` = your Copernicus S3 **secret key**
3. `Runtime` -> `Run all`. First run takes a few minutes (weights + a ~2 GB scene).


## 1. Clone the model + pull the weights (Git LFS)


In [ ]:
!sudo apt-get -qq update >/dev/null && sudo apt-get -qq install -y git-lfs >/dev/null
!git lfs install >/dev/null
import os
if not os.path.isdir('/content/vessel-detection-sentinels'):
    !git clone --depth 1 https://github.com/allenai/vessel-detection-sentinels.git /content/vessel-detection-sentinels
%cd /content/vessel-detection-sentinels
# pull only the Sentinel-1 detector + attribute weights + backbones
!git lfs pull --include='data/model_artifacts/sentinel-1/**' --include='torch_weights/**'
!ls -la data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445/best.pth data/model_artifacts/sentinel-1/attr/c34aa37/best.pth


## 2. Install dependencies
Uses Colab's GPU PyTorch; installs GDAL matched to the system lib, plus the geo + model deps.


In [ ]:
!sudo apt-get -qq install -y gdal-bin libgdal-dev >/dev/null
import subprocess
gdal_ver = subprocess.check_output(['gdal-config','--version']).decode().strip()
!pip -q install "GDAL=={}".format(gdal_ver) rasterio pyproj scikit-image lightgbm s2cloudless mapcalc python-dateutil boto3
import torch, torchvision
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available(), '| torchvision', torchvision.__version__)


## 3. Download a Sentinel-1 scene from Copernicus (S3)
Defaults to the Galapagos scene from 2026-06-27. To target another reserve/date, find a scene at https://browser.dataspace.copernicus.eu (Sentinel-1, GRD, IW) and paste its `.SAFE` name + S3 date path below.


In [ ]:
from google.colab import userdata
import boto3, pathlib
AK = userdata.get('CDSE_S3_KEY'); SK = userdata.get('CDSE_S3_SECRET')
s3 = boto3.client('s3', endpoint_url='https://eodata.dataspace.copernicus.eu',
                  aws_access_key_id=AK, aws_secret_access_key=SK)

SCENE = 'S1D_IW_GRDH_1SDV_20260627T114116_20260627T114145_003421_006075_6C48.SAFE'
S3_PREFIX = 'Sentinel-1/SAR/IW_GRDH_1S/2026/06/27/' + SCENE

dest = pathlib.Path('data') / SCENE
n = 0
for page in s3.get_paginator('list_objects_v2').paginate(Bucket='eodata', Prefix=S3_PREFIX):
    for obj in page.get('Contents', []):
        rel = obj['Key'][len(S3_PREFIX)+1:]
        out = dest / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file('eodata', obj['Key'], str(out))
        n += 1
print('downloaded', n, 'files ->', dest)
!du -sh {dest}


## 4. Run inference (GPU)


In [ ]:
!python src/main.py \
  --raw_path=data/ \
  --scratch_path=data/scratch/ \
  --output=data/output/ \
  --detector_model_dir=data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445 \
  --postprocess_model_dir=data/model_artifacts/sentinel-1/attr/c34aa37 \
  --scene_id={SCENE} \
  --conf=.85 --nms_thresh=10 --save_crops=False --catalog=sentinel1


## 5. Review + download the detections
Download `seavigil_detections.csv`, then hand it to SeaVigil's converter (`scripts/sar_detections_to_incidents.py`) to fold into the map with jurisdiction + authorization + evidence.


In [ ]:
import pandas as pd, glob
found = glob.glob('data/output/**/predictions.csv', recursive=True)
if not found:
    print('No predictions.csv found - check the inference cell output for errors.')
else:
    df = pd.read_csv(found[0])
    df['scene_id'] = SCENE
    print(len(df), 'vessel detections in', SCENE)
    cols = [c for c in ['lon','lat','score','vessel_length_m','vessel_speed_k','is_fishing_vessel'] if c in df.columns]
    display(df[cols].head(25))
    df.to_csv('seavigil_detections.csv', index=False)
    from google.colab import files; files.download('seavigil_detections.csv')
